In [1]:
import numpy as np
import pandas as pd
import torch
import os
import cv2
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [9]:
def iou(box1, box2): # box = [x_center, y_center, width, height]
    x11, y11, x12, y12 = box1[0] - box1[2]/2, box1[1] - box1[3]/2, box1[0]  + box1[2]/2, box1[1] + box1[3]/2
    x21, y21, x22, y22 = box2[0] - box2[2]/2, box2[1] - box2[3]/2, box2[0]  + box2[2]/2, box2[1] + box2[3]/2

    x_inter1, y_inter1 = max(x11, x21), max(y11, y21)
    x_inter2, t_inter2 = min(x12, x22), min(y12, y22)

    intersection = abs(x_inter2 - x_inter1) * abs(y_inter1 - t_inter2)

    box1_area, box2_area = box1[2] * box1[3], box2[2] * box2[3]
    
    return intersection / (box1_area + box2_area - intersection)

In [12]:
box1 = torch.tensor([1, 1, 2, 2])
box2 = torch.tensor([2, 1, 2, 2])
print(iou(box1, box2))

tensor(0.3333)


In [42]:
class BackDataSet(Dataset):
    def __init__(self, img_dir, label_dir, Anchor, img_size, S=[13, 26, 52], num_classes=14, transform=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.Anchor = Anchor
        self.img_size = img_size
        self.S = S
        self.num_classes = num_classes
        self.transform = transform

        self.imgs = sorted([img for img in os.listdir(img_dir) if img.endswith('.png')])
        self.labels = sorted([label for label in os.listdir(label_dir) if label.endswith('.txt')])

    def __len__(self):
        return len(self.imgs)
    
    def __getitem__(self, index):
        img_path = os.path.join(self.img_dir, self.imgs[index])
        label_path = os.path.join(self.label_dir, self.labels[index])
        annotation = np.loadtxt(label_path, delimiter=' ')
        img = np.array(cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB))
        img = cv2.resize(img, (self.img_size, self.img_size))
        annotation = self.resize_annotation(annotation, img.shape[0], self.img_size)
        if self.transform:
            img = self.transform(img)
            annotation = self.transform(annotation)
        
        target = [torch.zeros((len(self.Anchor) // 3, S, S, 6)) for S in self.S] # 6 = [Po, x, y, w, h, class]
        for box in annotation: # box = [class, x_topleft, y_topleft, width, height]
            class_label, x, y, width, height = box.tolist()
            iou_anchors = [iou([0, 0,width * self.img_size, height * self.img_size], [0,0] + anchor.tolist() ) for anchor in self.Anchor]
            iou_anchors = torch.tensor(iou_anchors)
            anchor_indices = iou_anchors.argsort(descending=True)
            anchored = [False, False, False]
            for anchor_idx in anchor_indices:
                S = self.S[anchor_idx // 3] # for example anchor_idex = 5//3 = 1 -> S = 26
                cell_x, cell_y = int(S * x), int(S * y) # for example x = 0.3, S = 26 -> cell_x = 7, y = 0.7, S = 26 -> cell_y = 18
                if not target[anchor_idx // 3][anchor_idx % 3, cell_y, cell_x, 0]: # if that cell is not already responsible for an object
                    if anchored[anchor_idx // 3]: # if the object is already assigned to another anchor
                        continue
                    anchored[anchor_idx // 3] = True # assign the object to the anchor
                    target[anchor_idx // 3][anchor_idx % 3, cell_y, cell_x, 0] = 1 # set the objectness score to 1
                    target[anchor_idx // 3][anchor_idx % 3, cell_y, cell_x, 5] = int(class_label) # set the class label
                    target[anchor_idx // 3][anchor_idx % 3, cell_y, cell_x, 1:5] = torch.tensor([x * S - cell_x, y * S - cell_y, width * S, height * S]) # x = 0.3, S = 26, cell_x = 7 -> x * S - cell_x = 0.8(percentage inside the cell), y = 0.7, S = 26, cell_y = 18 -> y * S - cell_y = 0.6(percentage inside the cell)
        return img, tuple(target)

    def resize_annotation(self, annotation, original_size, new_size):
        size_ratio = new_size / original_size
        resized_annotation = annotation.copy()
        resized_annotation[:, 1:] = size_ratio * annotation[:, 1:]
        return resized_annotation



In [43]:
anchors = torch.tensor([[0.03285871, 0.03285871], [0.07231843, 0.07231843], [ 0.08853575,  0.08853575], 
                        [0.03790532, 0.03790532], [0.04916684, 0.04916684], [0.02724177, 0.02724177], [0.0631068, 0.0631068], [0.04351664, 0.04351664], [0.0558598, 0.0558598]])

In [44]:
print(anchors.shape)

torch.Size([9, 2])


In [51]:
dataset = BackDataSet('dataset', 'dataset/annotation_percentage', anchors, 416, transform=None)
test_img, anchoredData = dataset[0]
print(test_img.shape)
print(len(anchoredData))
print(anchoredData[2].shape) # 3 boxes per scale, 52x52 grid, 6 = [Po, x, y, w, h, class]

# for box in test_label:
#     class_id, x1, y1, w, l = box
#     x2, y2 = x1 + w, y1 + l
#     cv2.rectangle(test_img, (int(x1*test_img.shape[0]), int(y1*test_img.shape[0])), (int(x2*test_img.shape[0]), int(y2*test_img.shape[0])), (0, 255, 0), 2)
#     cv2.putText(test_img, str(int(class_id)), (int(x1), int(y1)), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 1)
# plt.imshow(test_img)


(416, 416, 3)
3
torch.Size([3, 52, 52, 6])
